In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# Make a request

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)
print(message)
print(message.content[0].text)

In [ ]:
# Make a second request (without adding previous message)

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentence."
        }
    ]
)
print(message)
print(message.content[0].text)

In [ ]:
#Helper functions for Multi-Turn conversations

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [ ]:
# One basic multi-turn conversation

# Start with an empty message list
messages = []

print(f"First empty message:{messages}\n")

# Add in the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

print(f"First user message:{messages}\n")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

print(f"Messages with user and assistant response:{messages}\n")

# Add a follow-up question
add_user_message(messages, "Write another sentence")

print(f"Messages with an additional user message:{messages}\n")

# Get the follow-up response with full context
final_answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, final_answer)

print(f"Messages with the second assistant message:{messages}\n")


In [ ]:
# Chat exercise

messages = []

# Use a 'While True' loop to run the chatbot forever

while True:
    # Get user input
    user_input = input("> ")
    print(">", user_input)

    # Add user input to the list of messages
    add_user_message(messages, user_input)
    # Call Claude with the 'chat' function
    answer = chat(messages)
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    # Print the generated text
    print("---")
    print(answer)
    print("---")


In [ ]:
# New helper function to add system prompts


def chat(messages, system=None):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Quick test to generate message with system prompt

system_prompt=""""
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = []

print(f"First empty message:{messages}\n")

# Add in the initial user question
add_user_message(messages, "How do I solve 5x+3=2 for x?")

print(f"First user message:{messages}\n")

# Get Claude's response without the system prompt
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

print(f"Messages with user and assistant response without system prompt:{messages}\n")

messages = []

print(f"First empty message:{messages}\n")

# Add in the initial user question
add_user_message(messages, "How do I solve 5x+3=2 for x?")

print(f"First user message:{messages}\n")
# Get Claude's response wit the system prompt
answer = chat(messages,system_prompt)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

print(f"Messages with user and assistant response with system prompt:{messages}\n")


In [ ]:
# New helper function to add temperature + system prompts


def chat(messages, system=None, temperature=1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Quick test to generate message with adjusted temperature

messages = []

print(f"First empty message:{messages}\n")

# Add in the initial user question
add_user_message(messages, "Generate a one sentence movie idea")

print(f"First user message:{messages}\n")

# Get Claude's response without default temperature
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

print(f"Messages with user and assistant response with default temp:{messages}\n")

messages = []

print(f"First empty message:{messages}\n")

# Add in the initial user question
add_user_message(messages, "Generate a one sentence movie idea")

print(f"First user message:{messages}\n")

# Get Claude's response without temperature at 0.0
answer = chat(messages, temperature=0.0)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

print(f"Messages with user and assistant response with temp at 0.0:{messages}\n")


In [ ]:
# Basic streaming implementation

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

In [ ]:
# Alternative simplified text streaming

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

final_stream = stream.get_final_message()
print (f"Final stream:{final_stream}")


In [ ]:
# Controlling Model Output: 1) Message pre-filling:
messages = []
add_user_message(messages, "Is tea or coffee better at breakfast?")
add_assistant_message(messages, "Coffee is better because")
answer = chat(messages)
print(answer)

In [ ]:
# Controlling Model Output: 2) Stop sequences:

# Adapting the chat helper function:

# New helper function to add temperature + system prompts


def chat(messages, system=None, temperature=1.0, stop_sequences=None):

    if stop_sequences is None:
        stop_sequences = []
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Testing Stop sequences:

messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=["5"])

print(answer)

In [ ]:
# Structured data : json

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)

In [ ]:
# Structured data exercise (1st part)

messages = []

prompt="""
Generate three different sample AWS CLI commands.
"""
add_user_message(messages, prompt)

text = chat(messages)
text.strip()


In [ ]:
# Structured data exercise (2nd part)

from IPython.display import Markdown

Markdown(text)

In [ ]:
# Exercise:
# 1) Use message prefilling and stop sequences only to get three different commands
#   in a single response
# 2) There shouldn't be any comments or explanation
# 3) Hint: message prefilling isn't limited to just characters like ````

messages = []

user_prompt="""
Generate three different sample AWS CLI commands. Each should be very short.
"""
assistant_prompt="""
Here are all three commands in a single block without any comments ```bash"""

add_user_message(messages, user_prompt)

add_assistant_message(messages, assistant_prompt)

text = chat(messages, stop_sequences=["```"])
text.strip()


